# Building AI Agent

## Goal

This notebook is a hands-on journey to build an AI agent from scratch.

Each version introduces one new concept, allowing the agent to evolve step by step while practicing AI agent development.

---

## Version 7

In this version, we introduce an LLM Router.

The Router no longer selects tools using manually defined synonyms and scores.
Instead, it uses a small pretrained language model to understand the user's request and select the most appropriate tool.

The available tools and their descriptions are provided by the Tool Registry.

Only the Router uses an LLM in this version.
The Parser remains rule-based.

The main goal of this version is to introduce language-model-based routing while keeping the rest of the architecture unchanged.

## 1. Imports

In [1]:
import re
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as transformers_logging
from huggingface_hub import logging as hf_logging
from huggingface_hub.utils import disable_progress_bars

### Local Language Model

Version 7 uses a small pretrained language model for routing.

The model runs locally inside the notebook.
It is downloaded from Hugging Face and does not require an external inference API.

In [2]:
# Local language model

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

transformers_logging.set_verbosity_error()
hf_logging.set_verbosity_error()
disable_progress_bars()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

model.eval()

print(f"Model loaded on: {device}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded on: cpu


## 2. Tools

In [3]:
# Tools

def greeting(name):
    """Greet the given name."""
    return f"Hello {name.title()}, nice to meet you!"


def addition(a, b):
    """Add two numbers."""
    return a + b


def subtraction(a, b):
    """Subtract two numbers."""
    return a - b


def multiplication(a, b):
    """Multiply two numbers."""
    return a * b


def division(a, b):
    """Divide two numbers."""
    if b == 0:
        return "Error: division by zero!"
    return a / b


def power(a, b):
    """Raise a number to a power."""
    return a ** b

### Tool Registry

The Tool Registry provides a central place to store and access the tools available to the agent.

Instead of keeping tool metadata and synonyms in separate structures, each tool is registered together with its function, parameters, description, and synonyms.

The main goal is to centralize tool management while keeping the tools themselves unchanged.

In [4]:
# Tool registry

tool_registry = {}


def register_tool(name, function, parameters, description, synonyms):
    tool_registry[name] = {
        "function": function,
        "parameters": parameters,
        "description": description,
        "synonyms": synonyms
    }

In [5]:
# Register tools

register_tool(
    name="greeting",
    function=greeting,
    parameters=["name"],
    description="Greet the user by name.",
    synonyms=["hello", "hi", "hey", "good morning", "good afternoon", "good evening"]
)

register_tool(
    name="addition",
    function=addition,
    parameters=["a", "b"],
    description="Add two numbers.",
    synonyms=["add", "addition", "sum", "plus", "+"]
)

register_tool(
    name="subtraction",
    function=subtraction,
    parameters=["a", "b"],
    description="Subtract two numbers.",
    synonyms=["subtract", "subtraction", "minus", "take away", "-"]
)

register_tool(
    name="multiplication",
    function=multiplication,
    parameters=["a", "b"],
    description="Multiply two numbers.",
    synonyms=["multiply", "multiplication", "times", "*"]
)

register_tool(
    name="division",
    function=division,
    parameters=["a", "b"],
    description="Divide two numbers.",
    synonyms=["divide", "division", "divided by", "/"]
)

register_tool(
    name="power",
    function=power,
    parameters=["a", "b"],
    description="Raise a number to a power.",
    synonyms=["power", "raise", "raised to", "**"]
)

## 3. Parser

In [6]:
# Extract name

def extract_name(text):
    match = re.search(
        r"(?:my name is|i am|i'm|this is)\s+([A-Z][a-z]+)",
        text,
        re.IGNORECASE
    )

    if match:
        return match.group(1)

    words = re.findall(r"\b[A-Z][a-z]+\b", text)

    return words[1] if len(words) > 1 else None

In [7]:
# Extract numbers

def extract_numbers(text):
    numbers = []

    for token in text.split():
        token = re.sub(r"[^\w\s]", "", token)

        if token.isdigit():
            numbers.append(int(token))

    return numbers

In [8]:
# Parser

def build_parse_context(request):
    return {
        "numbers": extract_numbers(request),
        "name": extract_name(request)
    }


def extract_first_number(request, context):
    numbers = context["numbers"]
    return numbers[0] if len(numbers) > 0 else None


def extract_second_number(request, context):
    numbers = context["numbers"]
    return numbers[1] if len(numbers) > 1 else None


parameter_extractors = {
    "name": lambda request, context: context["name"],
    "a": extract_first_number,
    "b": extract_second_number
}


def parser(tool_name, request):
    if tool_name is None:
        return {}

    tool_info = tool_registry.get(tool_name)

    if not tool_info:
        return {}

    context = build_parse_context(request)
    arguments = {}

    for parameter in tool_info["parameters"]:
        extractor = parameter_extractors.get(parameter)

        if extractor is None:
            arguments[parameter] = None
        else:
            arguments[parameter] = extractor(request, context)

    return arguments

## 4. LLM Router

In [9]:
# LLM Router

def build_router_prompt(request):
    available_tools = []

    for tool_name, tool_info in tool_registry.items():
        available_tools.append(
            f"- {tool_name}: {tool_info['description']}"
        )

    tools_text = "\n".join(available_tools)
    allowed_answers = "\n".join(tool_registry.keys())

    return f"""
Select the tool that can actually handle the user request.

Available tools:
{tools_text}

User request:
{request}

Allowed answers:
{allowed_answers}
none

Select a tool only if it can directly perform the requested task.
If none of the available tools can handle the request, return none.
Do not select an unrelated tool just because it is the closest option.

Return only one allowed answer.
Do not explain your choice.
"""


def choose_tool(request):
    prompt = build_router_prompt(request)

    messages = [
        {
            "role": "system",
            "content": "You are a tool router. Select a tool only when it can handle the request. Otherwise output none."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=8, do_sample=False)

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()

    for tool_name in tool_registry:
        if re.search(rf"\b{re.escape(tool_name)}\b", response):
            return tool_name

    return None

## 5. Planner

In [10]:
# Planner

def planner(request):
    tool_name = choose_tool(request)

    if tool_name is None:
        return {
            "tool": None,
            "reason": "No suitable tool was found for the request.",
            "needs_arguments": []
        }

    tool_info = tool_registry.get(tool_name)

    return {
        "tool": tool_name,
        "reason": tool_info["description"],
        "needs_arguments": tool_info["parameters"]
    }

## 6. Executor

In [11]:
# Validate arguments

def arguments_are_valid(arguments):
    if not arguments:
        return False

    for value in arguments.values():
        if value is None:
            return False

    return True

In [12]:
# Executor

def executor(tool_name, arguments):
    tool_info = tool_registry.get(tool_name)

    if not tool_info:
        return "Tool not found!"

    if not arguments_are_valid(arguments):
        return "Invalid arguments"

    function = tool_info["function"]

    return function(**arguments)

## 7. Memory Manager

In [13]:
# Memory manager

memory = []


def save_to_memory(state):
    memory.append(state)


def get_memory():
    return memory


def get_last_interaction():
    if not memory:
        return None

    return memory[-1]


def clear_memory():
    memory.clear()

## 8. Agent

In [14]:
# Agent

def agent(request):
    state = {
        "request": request,
        "plan": None,
        "action": None,
        "arguments": {},
        "result": None
    }

    plan = planner(request)
    state["plan"] = plan

    action = plan["tool"]
    state["action"] = action

    if action is None:
        state["result"] = "I cannot handle this request yet."
        save_to_memory(state)
        return state

    arguments = parser(action, request)
    state["arguments"] = arguments

    result = executor(action, arguments)
    state["result"] = result

    save_to_memory(state)

    return state

## 9. Tests

In [15]:
clear_memory()

In [16]:
tool_registry

{'greeting': {'function': <function __main__.greeting(name)>,
  'parameters': ['name'],
  'description': 'Greet the user by name.',
  'synonyms': ['hello',
   'hi',
   'hey',
   'good morning',
   'good afternoon',
   'good evening']},
 'addition': {'function': <function __main__.addition(a, b)>,
  'parameters': ['a', 'b'],
  'description': 'Add two numbers.',
  'synonyms': ['add', 'addition', 'sum', 'plus', '+']},
 'subtraction': {'function': <function __main__.subtraction(a, b)>,
  'parameters': ['a', 'b'],
  'description': 'Subtract two numbers.',
  'synonyms': ['subtract', 'subtraction', 'minus', 'take away', '-']},
 'multiplication': {'function': <function __main__.multiplication(a, b)>,
  'parameters': ['a', 'b'],
  'description': 'Multiply two numbers.',
  'synonyms': ['multiply', 'multiplication', 'times', '*']},
 'division': {'function': <function __main__.division(a, b)>,
  'parameters': ['a', 'b'],
  'description': 'Divide two numbers.',
  'synonyms': ['divide', 'division', 

In [17]:
agent("What is 2 + 3?")

{'request': 'What is 2 + 3?',
 'plan': {'tool': 'addition',
  'reason': 'Add two numbers.',
  'needs_arguments': ['a', 'b']},
 'action': 'addition',
 'arguments': {'a': 2, 'b': 3},
 'result': 5}

In [18]:
agent("Hello, what is 2 + 3?")

{'request': 'Hello, what is 2 + 3?',
 'plan': {'tool': 'addition',
  'reason': 'Add two numbers.',
  'needs_arguments': ['a', 'b']},
 'action': 'addition',
 'arguments': {'a': 2, 'b': 3},
 'result': 5}

In [19]:
agent("Hello, my name is luca.")

{'request': 'Hello, my name is luca.',
 'plan': {'tool': 'greeting',
  'reason': 'Greet the user by name.',
  'needs_arguments': ['name']},
 'action': 'greeting',
 'arguments': {'name': 'luca'},
 'result': 'Hello Luca, nice to meet you!'}

In [20]:
agent("What is the weather in Rome?")

{'request': 'What is the weather in Rome?',
 'plan': {'tool': None,
  'reason': 'No suitable tool was found for the request.',
  'needs_arguments': []},
 'action': None,
 'arguments': {},
 'result': 'I cannot handle this request yet.'}

In [21]:
agent("Could you calculate the product of 2 and 3?")

{'request': 'Could you calculate the product of 2 and 3?',
 'plan': {'tool': 'multiplication',
  'reason': 'Multiply two numbers.',
  'needs_arguments': ['a', 'b']},
 'action': 'multiplication',
 'arguments': {'a': 2, 'b': 3},
 'result': 6}

In [22]:
get_memory()

[{'request': 'What is 2 + 3?',
  'plan': {'tool': 'addition',
   'reason': 'Add two numbers.',
   'needs_arguments': ['a', 'b']},
  'action': 'addition',
  'arguments': {'a': 2, 'b': 3},
  'result': 5},
 {'request': 'Hello, what is 2 + 3?',
  'plan': {'tool': 'addition',
   'reason': 'Add two numbers.',
   'needs_arguments': ['a', 'b']},
  'action': 'addition',
  'arguments': {'a': 2, 'b': 3},
  'result': 5},
 {'request': 'Hello, my name is luca.',
  'plan': {'tool': 'greeting',
   'reason': 'Greet the user by name.',
   'needs_arguments': ['name']},
  'action': 'greeting',
  'arguments': {'name': 'luca'},
  'result': 'Hello Luca, nice to meet you!'},
 {'request': 'What is the weather in Rome?',
  'plan': {'tool': None,
   'reason': 'No suitable tool was found for the request.',
   'needs_arguments': []},
  'action': None,
  'arguments': {},
  'result': 'I cannot handle this request yet.'},
 {'request': 'Could you calculate the product of 2 and 3?',
  'plan': {'tool': 'multiplication',

In [23]:
get_last_interaction()

{'request': 'Could you calculate the product of 2 and 3?',
 'plan': {'tool': 'multiplication',
  'reason': 'Multiply two numbers.',
  'needs_arguments': ['a', 'b']},
 'action': 'multiplication',
 'arguments': {'a': 2, 'b': 3},
 'result': 6}

## Notes

Some test cases were intentionally designed to verify specific parts of the agent.

- `"Hello, what is 2 + 3?"` checks that the LLM Router identifies addition as the main intent.
- `"Hello, my name is luca."` checks that the LLM Router selects the greeting tool while the existing rule-based Parser still extracts the name.
- `"Could you calculate the product of 2 and 3?"` checks that the LLM Router can understand the user's intent without depending on manually defined routing synonyms.
- `"What is the weather in Rome?"` checks that the LLM Router returns no tool when none of the registered tools can handle the request.
- The Tool Registry provides the available tool names and descriptions to the LLM Router.
- The language model is used only for tool selection.
- The Parser remains rule-based and continues to extract arguments using the existing parameter extractors.
- The Planner remains rule-based.
- The Executor, Memory Manager, and Agent remain unchanged.
- The agent still performs one action at a time.
- The language model runs locally inside the notebook and does not use an external inference API.

Future versions will introduce new components and gradually evolve the architecture.